# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [ ]:
import pandas as pd
house_df = pd.read_csv('house_sales.csv')
house_df['city'] = house_df['city'].replace('--', None)

missing_city = house_df['city'].isna().sum()
missing_city

np.int64(73)

# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [140]:
house_df = pd.read_csv('house_sales.csv')

clean_data = house_df

city_map = {
    '--':'Unknown'
}
clean_data['city'] = clean_data['city'].replace(city_map)
clean_data['city'] = clean_data['city'].astype('category')

clean_data['sale_date'] = pd.to_datetime(clean_data['sale_date'])
clean_data['months_listed'] = clean_data['months_listed'].fillna(round(clean_data['months_listed'].mean(),1))
clean_data['bedrooms'] = clean_data['bedrooms'].astype('category')
house_map = {
    'Det.' : 'Detached',
    'Terr.':'Terraced',
    'Semi' : 'Semi-detached'
} 
clean_data['house_type'] = clean_data['house_type'].replace(house_map)
clean_data['house_type'] = clean_data['house_type'].astype('category')

clean_data['area'] = clean_data['area'].str.replace(' sq.m.','').astype('float64')
clean_data['area'] = clean_data['area'].fillna(round(clean_data['area'].mean(),1))

# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [121]:
price_by_rooms = clean_data.groupby('bedrooms')['sale_price']\
                .agg(avg_price=lambda x: round(x.mean(), 1), var_price=lambda x: round(x.var(), 1))\
                .reset_index()

price_by_rooms

C:\Users\radus\AppData\Local\Temp\ipykernel_37860\3574019511.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_by_rooms = clean_data.groupby('bedrooms')['sale_price']\


,bedrooms,avg_price,var_price
0,2,67076.4,5.652896e+08
1,3,154665.1,2.378289e+09
2,4,234704.6,1.725211e+09
3,5,301515.9,2.484328e+09
4,6,375741.3,3.924432e+09


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


Train = pd.read_csv('train.csv')
Val = pd.read_csv('validation.csv')

X = Train.drop(['sale_price', 'sale_date'], axis=1)
y = Train['sale_price']
X = pd.get_dummies(X)
X_val = Val.drop(['sale_date'], axis=1)
X_val = pd.get_dummies(X_val)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)


label_e = LabelEncoder()
s_scaler = StandardScaler()

steps = [
    ("scaler", s_scaler),
    ("logreg", LinearRegression())
]
pipeline = Pipeline(steps)

pipeline.fit(X_train, y_train)


y_pred = pipeline.predict(X_test)

mse = MSE(y_test,y_pred)

print(mse**0.5)

y_pred = pipeline.predict(X_val)

base_result = pd.DataFrame()
base_result['house_id'] = X_val['house_id']
base_result['price'] = y_pred

base_result.head()

23461.811608549007


,house_id,price
0,1331375,121107.480694
1,1630115,302854.524684
2,1645745,384067.920827
3,1336775,124270.040606
4,1888274,270689.395025


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [123]:
from sklearn.ensemble import RandomForestRegressor

Train = pd.read_csv('train.csv')
Val = pd.read_csv('validation.csv')

X = Train.drop(['sale_price', 'sale_date'], axis=1)
y = Train['sale_price']
X = pd.get_dummies(X)
X_val = Val.drop(['sale_date'], axis=1)
X_val = pd.get_dummies(X_val)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)


rf = RandomForestRegressor(max_depth=10, 
                           max_features='sqrt',
                           n_estimators=350
                           )

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

rf_mse = MSE(y_test,y_pred)

print(rf_mse**0.5)

y_pred = rf.predict(X_val)

compare_result = pd.DataFrame()
compare_result['house_id'] = X_val['house_id']
compare_result['price'] = y_pred

compare_result.head()

15608.458505243561


,house_id,price
0,1331375,85169.603169
1,1630115,301385.960920
2,1645745,398299.547693
3,1336775,111597.112452
4,1888274,256684.385505
